# Stage 00a — PatSeer Download

**What this notebook does:**  
Opens a Chrome browser, logs into PatSeer, and iterates through every record in
your search result.  For each patent it clicks the *Drawings* tab and downloads
all available figures — three file types that PatSeer provides:

| Type | Example filename | Meaning |
|------|-----------------|---------|
| `img` | `US20220267016A1_img003.png` | Individual figure crop — PatSeer already separated these, no splitting needed |
| `D`   | `US20220267016A1_D00005.png` | Full drawing sheet — may contain multiple figures per page |
| `FAT` | `US20220267016A1_FAT001.png` | Composite sheet — may contain multiple figures per page |

Raw PatSeer filenames (e.g. `record_0002_US…-20220825-img003.png`) are cleaned
on save: the `record_NNNN_` prefix and `-YYYYMMDD-` date segment are stripped,
remaining hyphens become underscores.

A `{patent_id}_download_manifest.json` is written into each patent folder
recording which files were saved, the publication date, and the record position.

**Where it fits in the pipeline:**
```
00a  ←  YOU ARE HERE  (download from PatSeer)
 ↓
00b  (positional matching → _F / _Fu labels)
 ↓
02   (pad + resize to 518×518)
```

**To resume a partial run:** set `START_FROM` in Cell 1 to the last completed
record index and re-run.  Already-downloaded patents are skipped automatically.

---

| Cell | What it does |
|------|--------------|
| 1 | Load config; set `OUTPUT_DIR`, `START_FROM`, `END_AT` |
| 2 | Launch the downloader — opens Chrome, prompts for login, then loops all records |
| 3 | Scan `raw/` and print a per-type download coverage report |

In [ ]:
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
src_dir   = repo_root / "src"

# Load config_loader from an explicit absolute path — immune to sys.path ordering
# and sys.modules cache (the two bugs that keep breaking the plain import).
_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod        # register before exec so relative imports inside it work
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

print(f"config_loader loaded from: {_cl_path}")

# sys.path for the remaining imports (patseer_downloader, etc.)
for p in [str(repo_root), str(src_dir)]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(src_dir))

cfg = load_config()

# ─── Run parameters (edit here) ───────────────────────────────────────────────
OUTPUT_DIR  = cfg["paths"]["raw_images"]   # override with Path("...") if needed
START_FROM  = 1      # record index to start from (set higher to resume)
END_AT      = None   # last record to download; None = use TOTAL_RECORDS from script
# ──────────────────────────────────────────────────────────────────────────────

print(f"Output dir : {OUTPUT_DIR}")
print(f"Start from : {START_FROM}")
print(f"End at     : {END_AT or 'all records'}")

In [ ]:
# ── Migrate record_XXXX folders → patent-ID folders ──────────────────────────
# Run this once to fix folders downloaded before the URL-based patent ID fix.
# Safe to re-run: already-correct folders are skipped.
# If raw/ doesn't exist yet, prints a message and exits cleanly.

import re, json

_PAT_ID_RE = re.compile(r"([A-Z]{2}\d{7,}[A-Z0-9]+)", re.IGNORECASE)

def _patent_id_from_files(folder: Path) -> str | None:
    for f in sorted(folder.glob("*.png")):
        if re.search(r"_d\d{4,}", f.name, re.IGNORECASE):
            m = _PAT_ID_RE.search(f.name)
            if m:
                return m.group(1).upper()
    for f in sorted(folder.glob("*.png")):
        m = _PAT_ID_RE.search(f.name)
        if m:
            return m.group(1).upper()
    return None

raw_dir = OUTPUT_DIR
migrated, skipped, conflicts = [], [], []

if not raw_dir.exists():
    print(f"raw/ folder does not exist yet ({raw_dir}) — nothing to migrate.")
else:
    for folder in sorted(raw_dir.iterdir()):
        if not folder.is_dir() or not re.match(r"record_\d+$", folder.name):
            continue

        patent_id = _patent_id_from_files(folder)
        if not patent_id:
            print(f"  ? {folder.name} — could not determine patent ID, skipping")
            skipped.append(folder.name)
            continue

        dest_folder = raw_dir / patent_id
        if dest_folder.exists():
            print(f"  ⚠ {folder.name} → {patent_id} already exists (duplicate), skipping")
            conflicts.append((folder.name, patent_id))
            continue

        record_prefix = folder.name + "_"
        for f in list(folder.glob("*")):
            new_name = f.name
            if new_name.lower().startswith(record_prefix.lower()):
                new_name = new_name[len(record_prefix):]
            if new_name != f.name:
                f.rename(folder / new_name)

        old_manifest = folder / f"{folder.name}_download_manifest.json"
        if old_manifest.exists():
            data = json.loads(old_manifest.read_text())
            data["patent_id"] = patent_id
            for key in ("img_files", "d_files", "fat_files"):
                data[key] = [
                    n[len(record_prefix):] if n.lower().startswith(record_prefix.lower()) else n
                    for n in data.get(key, [])
                ]
            (folder / f"{patent_id}_download_manifest.json").write_text(json.dumps(data, indent=2))
            old_manifest.unlink()

        folder.rename(dest_folder)
        print(f"  ✓ {folder.name} → {patent_id}")
        migrated.append((folder.name, patent_id))

    print(f"\nMigrated: {len(migrated)}  |  Conflicts (skipped): {len(conflicts)}  |  Unknown: {len(skipped)}")

In [ ]:
import patseer_downloader as dl

# Patch module-level configuration before launching
dl.OUTPUT_DIR = OUTPUT_DIR
dl.START_FROM = START_FROM
if END_AT is not None:
    dl.TOTAL_RECORDS = END_AT

# A Chrome window will open — log in to PatSeer, then press Enter here.
# clean_filename() is applied to every saved file inside download_images().
dl.main()


In [ ]:
import re

raw_dir = OUTPUT_DIR

patents_attempted  = 0
patents_with_img   = 0
patents_with_d     = 0
patents_with_fat   = 0
total_img          = 0
total_d            = 0
total_fat          = 0

for patent_dir in sorted(raw_dir.iterdir()):
    if not patent_dir.is_dir():
        continue
    patents_attempted += 1

    imgs  = [p for p in patent_dir.glob("*.png") if "_img" in p.name.lower()]
    dfiles = [p for p in patent_dir.glob("*.png") if re.search(r"_d\d{4,}", p.name, re.IGNORECASE)]
    fats  = [p for p in patent_dir.glob("*.png") if "_fat" in p.name.lower()]

    if imgs:   patents_with_img += 1
    if dfiles: patents_with_d   += 1
    if fats:   patents_with_fat += 1

    total_img += len(imgs)
    total_d   += len(dfiles)
    total_fat += len(fats)

def _pct(n):
    return f"{n / patents_attempted * 100:.0f}%" if patents_attempted else "N/A"

avg_imgs = total_img / patents_with_img if patents_with_img else 0

print("══════════════════════════════════════")
print("Download Coverage Report")
print("══════════════════════════════════════")
print(f"Patents attempted      : {patents_attempted}")
print(f"Patents with img files : {patents_with_img}  ({_pct(patents_with_img)})")
print(f"Patents with D files   : {patents_with_d}  ({_pct(patents_with_d)})")
print(f"Patents with FAT files : {patents_with_fat}  ({_pct(patents_with_fat)})")
print(f"Total img files        : {total_img}")
print(f"Total D files          : {total_d}")
print(f"Total FAT files        : {total_fat}")
print(f"Average imgs/patent    : {avg_imgs:.1f}")
print("══════════════════════════════════════")